# Synthetic speakers with globi

For one speaker X, re-speak M of their clips in the voices of N donor speakers, in X's dialect, with the
[globi](../../../05_Tools/globi) anonymizer (XTTS v2 fine-tuned on Swiss German). The result is N × M
recordings with X's words and dialect but other voices, in `data/stt4sg-350_synth` (wavs + `synth.tsv`).

In [1]:
import io
import json
import tarfile
import time
import urllib.error
import urllib.request
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf

In [2]:
GLOBI_URL = "http://spark-b2f4.tail13dd45.ts.net:8010"
DATA_DIR = Path("../data/stt4sg-350_v2-1")
OUT_DIR = Path("../data/stt4sg-350_synth")
N_DONORS = 5            # N: voices
N_SAMPLES = 5           # M: clips of speaker X
REFERENCE_SECONDS = 20  # donor clips are joined to about this length (globi allows up to 60 s)
SEED = 1234

# dialect_region of STT4SG-350 -> globi language token
LANGUAGES = {"Basel": "ch_bs", "Bern": "ch_be", "Graubünden": "ch_gr", "Innerschweiz": "ch_in",
             "Ostschweiz": "ch_os", "Wallis": "ch_vs", "Zürich": "ch_zh"}

## Speakers

Only the test split is local (`test.tsv` + `clips__test.tar`); its mp3s are read straight from the tar.

In [3]:
meta = pd.read_csv(DATA_DIR / "test.tsv", sep="\t")
tar = tarfile.open(DATA_DIR / "clips__test.tar")
members = {m.name: m for m in tar.getmembers()}


def load_clip(path: str) -> tuple[np.ndarray, int]:
    audio, sr = sf.read(io.BytesIO(tar.extractfile(members[path]).read()), dtype="float32")
    return (audio.mean(1) if audio.ndim > 1 else audio), sr


def to_wav(audio: np.ndarray, sr: int) -> bytes:
    buf = io.BytesIO()
    sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    return buf.getvalue()


speakers = (meta.groupby("client_id")
            .agg(dialect_region=("dialect_region", "first"), canton=("canton", "first"),
                 zipcode=("zipcode", "first"), age=("age", "first"), gender=("gender", "first"),
                 n_clips=("path", "size"), hours=("duration", lambda d: d.sum() / 3600))
            .reset_index())
speakers.groupby("dialect_region").size()

dialect_region
Basel           12
Bern            12
Graubünden      11
Innerschweiz    11
Ostschweiz      10
Wallis           8
Zürich          12
dtype: int64

X is drawn at random; the N donors come from other dialect regions (one per region where possible), so
the dialect of the output can only come from the language token and X's words, not from the voice.

In [4]:
rng = np.random.default_rng(SEED)
speaker_x = speakers.iloc[rng.integers(len(speakers))]
language = LANGUAGES[speaker_x.dialect_region]

others = speakers[speakers.dialect_region != speaker_x.dialect_region]
regions = rng.permutation(others.dialect_region.unique())
donors = pd.DataFrame([others[others.dialect_region == r].sample(1, random_state=SEED).iloc[0]
                       for r in np.resize(regions, N_DONORS)])
if donors.client_id.duplicated().any():  # fewer regions than donors with no unique pick left
    donors = others.sample(N_DONORS, random_state=SEED)

print(f"X: {speaker_x.client_id} ({speaker_x.dialect_region}, {speaker_x.canton} {speaker_x.zipcode}, "
      f"{speaker_x.gender}, {speaker_x.age}) -> {language}")
donors[["client_id", "dialect_region", "canton", "gender", "age", "n_clips"]]

X: fb1d2127-e914-470f-b684-57c697eacc1c (Basel, BL 4123, female, thirties) -> ch_bs


,client_id,dialect_region,canton,gender,age,n_clips
17,3a231070-4b09-40eb-a319-1f3ed4e61ad9,Wallis,VS,female,twenties,363
43,887b50f8-215b-4a1d-8f32-13516da6506f,Bern,BE,female,sixties,358
54,aa7d9d2c-90d3-41f9-a92e-f07b93765f8b,Innerschweiz,ZG,female,twenties,363
45,8d167b1c-7f89-4487-a0a9-2bbe08bf0b41,Ostschweiz,TG,male,fourties,363
66,d2f85a8f-7c45-48e3-a804-0b192f7f8ad6,Graubünden,GR,male,twenties,648


In [5]:
samples = meta[meta.client_id == speaker_x.client_id].sample(N_SAMPLES, random_state=SEED)
samples[["path", "duration", "sentence"]]

,path,duration,sentence
4011,fb1d2127-e914-470f-b684-57c697eacc1c/7f1828ad5...,5.774667,Die Stadt ist via Dividende vom Erfolg der Eni...
22383,fb1d2127-e914-470f-b684-57c697eacc1c/84d2718b0...,6.206667,Im neuen Kalenderjahr soll dann auch etwas für...
18603,fb1d2127-e914-470f-b684-57c697eacc1c/5d5192f6c...,3.794667,Das geht in diesem Auto einfach nicht.
5972,fb1d2127-e914-470f-b684-57c697eacc1c/646bd23dc...,7.898667,Momentan bewegen wir uns bei geplanten 13.9 Mi...
2435,fb1d2127-e914-470f-b684-57c697eacc1c/8350be028...,4.298667,Verpackt ist das Ganze in eine schicke Oberflä...


## Donor references

XTTS conditions on the reference voice; a few joined clips give a steadier voice than a single short one.

In [6]:
def donor_reference(client_id: str) -> tuple[bytes, float]:
    clips = meta[meta.client_id == client_id].sample(frac=1, random_state=SEED)
    parts, total, sr = [], 0.0, None
    for path in clips.path:
        audio, clip_sr = load_clip(path)
        if sr is not None and clip_sr != sr:
            continue
        sr = clip_sr
        parts += [audio, np.zeros(int(0.2 * sr), dtype=np.float32)]
        total += len(audio) / sr
        if total >= REFERENCE_SECONDS:
            break
    audio = np.concatenate(parts[:-1])[: int(55 * sr)]
    return to_wav(audio, sr), len(audio) / sr


references = {cid: donor_reference(cid) for cid in donors.client_id}
{cid: f"{sec:.1f} s" for cid, (_, sec) in references.items()}

{'3a231070-4b09-40eb-a319-1f3ed4e61ad9': '21.7 s',
 '887b50f8-215b-4a1d-8f32-13516da6506f': '21.2 s',
 'aa7d9d2c-90d3-41f9-a92e-f07b93765f8b': '24.3 s',
 '8d167b1c-7f89-4487-a0a9-2bbe08bf0b41': '23.9 s',
 'd2f85a8f-7c45-48e3-a804-0b192f7f8ad6': '21.0 s'}

## Anonymizer calls

All N × M jobs are submitted at once (the gateway queues them for the workers), then polled. The
Standard German `sentence` is passed as `text`, so the worker skips Whisper.

In [7]:
def submit(target: bytes, reference: bytes, language: str, text: str) -> dict:
    boundary = uuid.uuid4().hex
    body = io.BytesIO()
    for name, value in [("language", language), ("text", text)]:
        body.write(f'--{boundary}\r\nContent-Disposition: form-data; name="{name}"\r\n\r\n{value}\r\n'.encode())
    for name, data in [("target_wav", target), ("reference_wav", reference)]:
        body.write(f'--{boundary}\r\nContent-Disposition: form-data; name="{name}"; filename="{name}.wav"\r\n'
                   f"Content-Type: audio/wav\r\n\r\n".encode() + data + b"\r\n")
    body.write(f"--{boundary}--\r\n".encode())
    request = urllib.request.Request(f"{GLOBI_URL}/jobs", data=body.getvalue(), method="POST",
                                     headers={"Content-Type": f"multipart/form-data; boundary={boundary}"})
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.load(response)
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"globi refused the job ({e.code}): {e.read().decode()}") from None


def get(path: str) -> bytes:
    with urllib.request.urlopen(f"{GLOBI_URL}{path}", timeout=60) as response:
        return response.read()


print(json.loads(get("/health")))

{'status': 'ok', 'workers': 2, 'queued': 0}


In [8]:
targets = {row.path: to_wav(*load_clip(row.path)) for row in samples.itertuples()}
jobs = []
for donor in donors.itertuples():
    for sample in samples.itertuples():
        job = submit(targets[sample.path], references[donor.client_id][0], language, sample.sentence)
        jobs.append({"job_id": job["id"], "donor": donor.client_id, "source_path": sample.path})
print(f"submitted {len(jobs)} jobs")

submitted 25 jobs


In [9]:
pending = {job["job_id"] for job in jobs}
status = {}
start = time.time()
while pending:
    for job_id in list(pending):
        status[job_id] = json.loads(get(f"/jobs/{job_id}"))
        if status[job_id]["status"] in ("completed", "failed"):
            pending.discard(job_id)
    print(f"\r{len(jobs) - len(pending)}/{len(jobs)} done after {time.time() - start:.0f} s", end="")
    if pending:
        time.sleep(5)

failed = {job_id: s.get("error") for job_id, s in status.items() if s["status"] == "failed"}
print("\nfailed:", failed or "none")

25/25 done after 1 s
failed: none


## Save

Layout as in STT4SG-350: `clips/<client_id>/<id>.wav` and a tsv with its columns. Each donor × X pair is
a new synthetic speaker (`client_id`); the location labels (`dialect_region`, `canton`, `zipcode`) are
X's, `age` and `gender` the donor's (the voice). The `source_*` / `donor_*` columns link back to the
original clips.

In [10]:
synth_ids = {cid: str(uuid.uuid5(uuid.NAMESPACE_URL, f"{speaker_x.client_id}/{cid}")) for cid in donors.client_id}
donor_info = donors.set_index("client_id")
source_info = samples.set_index("path")

rows = []
for job in jobs:
    s = status[job["job_id"]]
    if s["status"] != "completed":
        continue
    src, donor = source_info.loc[job["source_path"]], donor_info.loc[job["donor"]]
    path = f"{synth_ids[job['donor']]}/{job['job_id']}.wav"
    (OUT_DIR / "clips" / path).parent.mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "clips" / path).write_bytes(get(f"/jobs/{job['job_id']}/result.wav"))
    rows.append({
        "path": path, "duration": sf.info(OUT_DIR / "clips" / path).duration, "sentence": src.sentence,
        "sentence_source": src.sentence_source, "client_id": synth_ids[job["donor"]],
        "dialect_region": speaker_x.dialect_region, "canton": speaker_x.canton, "zipcode": speaker_x.zipcode,
        "age": donor.age, "gender": donor.gender,
        "language": language, "transcript": s.get("transcript", ""), "job_id": job["job_id"],
        "source_path": job["source_path"], "source_client_id": speaker_x.client_id,
        "donor_client_id": job["donor"], "donor_dialect_region": donor.dialect_region,
        "donor_canton": donor.canton, "donor_zipcode": donor.zipcode,
    })

synth = pd.DataFrame(rows)
tsv = OUT_DIR / "synth.tsv"
if tsv.exists():  # append to earlier runs (other X), replacing rows of the same files
    synth = pd.concat([pd.read_csv(tsv, sep="\t"), synth]).drop_duplicates("path", keep="last")
synth.to_csv(tsv, sep="\t", index=False)
print(f"{len(rows)} recordings -> {OUT_DIR}, {len(synth)} in {tsv.name}")
synth.tail(len(rows))[["client_id", "donor_dialect_region", "duration", "sentence", "transcript"]]

25 recordings -> ../data/stt4sg-350_synth, 50 in synth.tsv


,client_id,donor_dialect_region,duration,sentence,transcript
0,e41459c3-4452-5e20-912a-ca2a64a2af55,Wallis,5.802667,Die Stadt ist via Dividende vom Erfolg der Eni...,Die Stadt ist via Dividende vom Erfolg der Eni...
1,e41459c3-4452-5e20-912a-ca2a64a2af55,Wallis,6.261333,Im neuen Kalenderjahr soll dann auch etwas für...,Im neuen Kalenderjahr soll dann auch etwas für...
2,e41459c3-4452-5e20-912a-ca2a64a2af55,Wallis,3.850667,Das geht in diesem Auto einfach nicht.,Das geht in diesem Auto einfach nicht.
3,e41459c3-4452-5e20-912a-ca2a64a2af55,Wallis,7.936000,Momentan bewegen wir uns bei geplanten 13.9 Mi...,Momentan bewegen wir uns bei geplanten 13.9 Mi...
4,e41459c3-4452-5e20-912a-ca2a64a2af55,Wallis,4.362667,Verpackt ist das Ganze in eine schicke Oberflä...,Verpackt ist das Ganze in eine schicke Oberflä...
5,7c65c93b-4827-583e-9552-bca6dfa36e62,Bern,5.802667,Die Stadt ist via Dividende vom Erfolg der Eni...,Die Stadt ist via Dividende vom Erfolg der Eni...
6,7c65c93b-4827-583e-9552-bca6dfa36e62,Bern,6.261333,Im neuen Kalenderjahr soll dann auch etwas für...,Im neuen Kalenderjahr soll dann auch etwas für...
7,7c65c93b-4827-583e-9552-bca6dfa36e62,Bern,3.850667,Das geht in diesem Auto einfach nicht.,Das geht in diesem Auto einfach nicht.
8,7c65c93b-4827-583e-9552-bca6dfa36e62,Bern,7.936000,Momentan bewegen wir uns bei geplanten 13.9 Mi...,Momentan bewegen wir uns bei geplanten 13.9 Mi...
9,7c65c93b-4827-583e-9552-bca6dfa36e62,Bern,4.362667,Verpackt ist das Ganze in eine schicke Oberflä...,Verpackt ist das Ganze in eine schicke Oberflä...


## Listen

Original clip of X next to its re-spoken versions.

In [11]:
from IPython.display import Audio, display

for source_path, group in synth.tail(len(rows)).groupby("source_path"):
    print(group.sentence.iloc[0])
    audio, sr = load_clip(source_path)
    display(Audio(audio, rate=sr))
    for row in group.itertuples():
        print(f"  voice of {row.donor_client_id[:8]} ({row.donor_dialect_region})")
        display(Audio(str(OUT_DIR / "clips" / row.path)))
    break  # first sample only

Das geht in diesem Auto einfach nicht.


  voice of 3a231070 (Wallis)


  voice of 887b50f8 (Bern)


  voice of aa7d9d2c (Innerschweiz)


  voice of 8d167b1c (Ostschweiz)


  voice of d2f85a8f (Graubünden)
